# Open Story - Event Stream Explorer

Interactive analysis of Claude agent event data.
Loads raw JSONL transcripts from `~/.claude/projects/` directly.

```bash
cd scripts
uv run --with pandas --with plotly --with jupyter jupyter lab explore.ipynb
```

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

from load_transcripts import load_all

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

## 1. Load data

In [ ]:
# Load last 24h (increase hours= for more history)
records = load_all(hours=24)
df = pd.DataFrame(records)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Records: {len(df):,}")
print(f"Sessions: {df['session_id'].nunique()}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## 2. Event type distribution

In [ ]:
type_counts = df['record_type'].value_counts()
print(type_counts)

fig = px.pie(values=type_counts.values, names=type_counts.index,
             title='Record Type Distribution',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
# Subtype breakdown (progress types, system types)
subtype_counts = df[df['subtype'] != '']['subtype'].value_counts().head(20)
print(subtype_counts)

fig = px.bar(x=subtype_counts.values, y=subtype_counts.index,
             orientation='h', title='Top Subtypes',
             labels={'x': 'Count', 'y': 'Subtype'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig.show()

## 3. Tool usage

In [ ]:
tools = df[df['record_type'] == 'tool_call']
tool_counts = tools['tool_name'].value_counts()
print(f"Tool calls: {len(tools):,}\n")
print(tool_counts)

fig = px.bar(x=tool_counts.values, y=tool_counts.index,
             orientation='h', title='Tool Usage Frequency',
             labels={'x': 'Count', 'y': 'Tool'},
             color=tool_counts.index)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

In [ ]:
# What are agents reading/editing/bashing?
for tool in ['Bash', 'Read', 'Edit', 'Grep']:
    subset = tools[tools['tool_name'] == tool]
    if len(subset) > 0:
        print(f"\n--- {tool} ({len(subset)} calls) ---")
        summaries = subset['tool_input_summary'].value_counts().head(10)
        for s, c in summaries.items():
            print(f"  {c:>4}x  {str(s)[:80]}")

## 4. Timeline

In [ ]:
# Events over time by record_type (1-minute buckets)
df_ts = df.dropna(subset=['timestamp']).copy()
df_ts['minute'] = df_ts['timestamp'].dt.floor('1min')

heatmap = df_ts.groupby(['minute', 'record_type']).size().reset_index(name='count')

fig = px.scatter(heatmap, x='minute', y='record_type', size='count',
                 color='record_type', title='Event Timeline (1-min buckets)',
                 labels={'minute': 'Time', 'record_type': 'Type', 'count': 'Events'})
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# Activity bursts: events per minute (all types)
activity = df_ts.groupby('minute').size().reset_index(name='events')

fig = px.area(activity, x='minute', y='events',
              title='Activity Over Time (events/min)',
              labels={'minute': 'Time', 'events': 'Events per minute'})
fig.show()

## 5. Event sequences

In [ ]:
# Bigrams: what follows what?
types_seq = df.sort_values('timestamp')['record_type'].tolist()
bigrams = Counter(zip(types_seq, types_seq[1:]))

print("Top 15 event pairs (A -> B):")
for (a, b), count in bigrams.most_common(15):
    print(f"  {a:20s} -> {b:20s}  {count:>6,}")

In [ ]:
# Sankey: event flow
# Filter to interesting types (skip progress self-loops for readability)
interesting = [p for p, c in bigrams.most_common(30) if p[0] != p[1]]
top_transitions = [(p, bigrams[p]) for p in interesting[:20]]

if top_transitions:
    all_types = sorted(set(t for pair, _ in top_transitions for t in pair))
    idx = {t: i for i, t in enumerate(all_types)}
    
    fig = go.Figure(go.Sankey(
        node=dict(label=all_types, pad=15, thickness=20),
        link=dict(
            source=[idx[a] for (a, _), _ in top_transitions],
            target=[idx[b] for (_, b), _ in top_transitions],
            value=[c for _, c in top_transitions],
        )
    ))
    fig.update_layout(title='Event Flow (non-self transitions)', height=500)
    fig.show()

## 6. Sessions

In [ ]:
session_stats = df[df['session_id'] != ''].groupby('session_id').agg(
    events=('uuid', 'count'),
    tool_calls=('record_type', lambda x: (x == 'tool_call').sum()),
    errors=('is_error', 'sum'),
    start=('timestamp', 'min'),
    end=('timestamp', 'max'),
).sort_values('events', ascending=False)

session_stats['duration_min'] = (session_stats['end'] - session_stats['start']).dt.total_seconds() / 60
session_stats['tools_pct'] = (session_stats['tool_calls'] / session_stats['events'] * 100).round(1)

print(f"Sessions: {len(session_stats)}")
session_stats.head(15)

In [ ]:
fig = px.histogram(session_stats, x='events', nbins=30,
                   title='Session Size Distribution',
                   labels={'events': 'Events', 'count': 'Sessions'})
fig.show()

## 7. Drill into a session

In [ ]:
# Pick the largest session (or paste a specific ID)
sid = session_stats.index[0]
# sid = "paste-session-id-here"

sess = df[df['session_id'] == sid].sort_values('timestamp')
print(f"Session: {sid[:20]}...")
print(f"Events: {len(sess):,}")
print(f"Duration: {sess['timestamp'].max() - sess['timestamp'].min()}")
print()
print(sess['record_type'].value_counts())
print()
print("Tools:")
print(sess[sess['record_type'] == 'tool_call']['tool_name'].value_counts())

In [ ]:
# Session timeline: tool calls, messages, and reasoning over time
interesting_types = ['tool_call', 'user_message', 'assistant_message', 'reasoning']
sess_events = sess[sess['record_type'].isin(interesting_types)].copy()

if len(sess_events) > 0:
    fig = px.scatter(sess_events, x='timestamp', y='record_type',
                     color='tool_name',
                     hover_data=['tool_input_summary', 'text'],
                     title=f'Session: {sid[:16]}...',
                     labels={'timestamp': 'Time', 'record_type': 'Type'})
    fig.update_traces(marker=dict(size=8))
    fig.update_layout(height=350)
    fig.show()

In [ ]:
# Turn durations from system events
turns = sess[(sess['record_type'] == 'system_event') & (sess['subtype'] == 'turn_duration')].copy()
if len(turns) > 0:
    turns['duration_s'] = turns['duration_ms'] / 1000
    print(f"Turns: {len(turns)}")
    print(f"Total time: {turns['duration_s'].sum():.0f}s")
    print(f"Mean: {turns['duration_s'].mean():.1f}s, Max: {turns['duration_s'].max():.1f}s")
    
    fig = px.bar(turns, x='timestamp', y='duration_s',
                 title='Turn Durations',
                 labels={'timestamp': 'Time', 'duration_s': 'Seconds'})
    fig.show()
else:
    print("No turn duration data.")

## 8. Call → Result pairing and code identification

Every tool_result can be matched back to its originating tool_call via `tool_id`.
This lets us identify what kind of output a result contains (file contents, git diff, command output, etc.).

In [ ]:
# Pair every tool_result back to its originating tool_call
calls = df[df['record_type'] == 'tool_call'].set_index('tool_id')
results = df[df['record_type'] == 'tool_result'].copy()
results['origin_tool'] = results['tool_id'].map(calls['tool_name'])
results['origin_summary'] = results['tool_id'].map(calls['tool_input_summary'])

paired = results['origin_tool'].value_counts()
print(f"Tool results matched to calls: {results['origin_tool'].notna().sum()}/{len(results)}")
print()
print("Results by originating tool:")
print(paired)

In [ ]:
# Bash command categorization — what kinds of shell commands does the agent run?
bash_calls = df[(df['record_type'] == 'tool_call') & (df['tool_name'] == 'Bash')].copy()

def categorize_bash(cmd):
    cmd = str(cmd).lower()
    if 'git ' in cmd: return 'git'
    if 'cargo ' in cmd: return 'cargo'
    if 'npm ' in cmd or 'npx ' in cmd: return 'npm/npx'
    if 'docker' in cmd: return 'docker'
    if 'python ' in cmd: return 'python'
    if 'curl ' in cmd: return 'curl'
    if 'cat ' in cmd or 'head ' in cmd or 'tail ' in cmd: return 'file-read'
    if 'ls ' in cmd or 'find ' in cmd: return 'file-list'
    if 'grep ' in cmd or 'rg ' in cmd: return 'search'
    return 'other'

bash_calls['bash_category'] = bash_calls['tool_input_summary'].apply(categorize_bash)
cat_counts = bash_calls['bash_category'].value_counts()
print(f"Bash calls: {len(bash_calls):,}\n")
print(cat_counts)

fig = px.pie(values=cat_counts.values, names=cat_counts.index,
             title='Bash Command Categories',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
# Code identification: which tool results contain highlightable code?
# Strategy: pair result -> call, then infer language from the call's context

EXT_MAP = {
    '.ts': 'typescript', '.tsx': 'tsx', '.js': 'javascript', '.jsx': 'jsx',
    '.rs': 'rust', '.py': 'python', '.json': 'json', '.yaml': 'yaml',
    '.yml': 'yaml', '.toml': 'toml', '.md': 'markdown', '.sh': 'bash',
    '.css': 'css', '.html': 'html', '.sql': 'sql', '.go': 'go',
}

def detect_result_language(row):
    """Infer what language a tool_result contains based on its originating call."""
    tool = row.get('origin_tool', '')
    summary = str(row.get('origin_summary', ''))
    
    if tool == 'Read':
        # File contents — use extension
        for ext, lang in EXT_MAP.items():
            if summary.endswith(ext):
                return lang
        return 'text'
    elif tool == 'Bash':
        if 'git diff' in summary or 'git show' in summary:
            return 'diff'
        if 'git log' in summary:
            return 'git-log'
        return 'shell-output'
    elif tool == 'Grep':
        return 'grep-matches'
    elif tool == 'Glob':
        return 'file-paths'
    elif tool in ('Edit', 'Write'):
        return 'status-message'
    return 'unknown'

results['detected_lang'] = results.apply(detect_result_language, axis=1)
lang_counts = results['detected_lang'].value_counts()
print("Detected language/content type in tool results:")
print(lang_counts)

fig = px.bar(x=lang_counts.values, y=lang_counts.index,
             orientation='h', title='Tool Result Content Types',
             labels={'x': 'Count', 'y': 'Content Type'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=400)
fig.show()

In [ ]:
# What files does the agent read most? (language distribution of file reads)
read_calls = df[(df['record_type'] == 'tool_call') & (df['tool_name'] == 'Read')].copy()

def ext_from_path(path):
    path = str(path)
    dot = path.rfind('.')
    if dot == -1: return '(none)'
    ext = path[dot:].lower()
    # Normalize common extensions
    return EXT_MAP.get(ext, ext)

read_calls['language'] = read_calls['tool_input_summary'].apply(ext_from_path)
read_lang = read_calls['language'].value_counts()
print(f"Read calls: {len(read_calls):,}\n")
print("Files read by language:")
print(read_lang)

print("\nTop 15 most-read files:")
top_files = read_calls['tool_input_summary'].value_counts().head(15)
for path, count in top_files.items():
    # Shorten path for display
    short = str(path).replace('/home/dev/projects/my-project/', '')
    short = short.replace('.claude/worktrees/hazy-sniffing-goose/', '')
    print(f"  {count:>3}x  {short[:70]}")